# SQL 기초 — 테이블 결합과 중첩 쿼리

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

# SQL 기초 — 테이블 결합과 중첩 쿼리

## 기본키·외래키 · JOIN · UNION · 서브쿼리 · EXISTS

---

이전 시간까지 우리는 *한 테이블 안*에서 조회하고 요약했습니다. 그런데 모두몰의 진짜 질문들은 테이블 하나로는 답할 수 없습니다. _"**고객 이름**별 매출은?"_ → 주문(`orders`)과 고객(`customers`)을 합쳐야 하고, _"**카테고리별** 매출은?"_ → 주문 상세(`order_items`)와 상품(`products`)을 합쳐야 합니다.

오늘은 흩어진 테이블을 **`JOIN`** 으로 잇고, 세로로 이어 붙이는 **`UNION`**, 그리고 쿼리 안에 쿼리를 넣는 **서브쿼리** 를 배웁니다. 드디어 데이터가 하나로 모입니다.

> 💬 **오늘의 한 문장**  
> "데이터는 여러 표로 나뉘어 있다. 질문에 답하려면 그것을 다시 이어야 한다."

## 📋 학습 목표

이 학습 노트북을 마치면 여러분은 다음을 할 수 있습니다.

1. 기본키(PK)·외래키(FK)로 테이블 사이의 관계를 이해할 수 있습니다.
2. `INNER`·`LEFT`·`RIGHT JOIN`으로 여러 테이블을 목적에 맞게 결합할 수 있습니다.
3. `UNION`/`UNION ALL`로 두 결과를 세로로 이어 붙일 수 있습니다.
4. 서브쿼리(`WHERE`절·`FROM`절)로 복잡한 질문을 단계로 나눠 풀 수 있습니다.

_💡 위 목록을 천천히 읽고, 지금 할 수 있는 것과 아직 낯선 것을 마음속으로 표시해 봅시다. 노트북을 마친 뒤 다시 돌아와 비교하면 성장이 눈에 보일 것입니다._

## 📚 학습 목차

| Part          | 주제                                           | 이 파트에서 답할 질문                          | 예상 시간            |
| ------------- | ---------------------------------------------- | ---------------------------------------------- | -------------------- |
| **0**         | 지난 시간 복습과 데이터 준비                   | 한 테이블 요약은 익숙해졌습니까?               | 15분                 |
| **1**         | 기본키(PK)와 외래키(FK)                        | 테이블은 왜 나뉘어 있습니까?                   | 25분                 |
| **2**         | `INNER JOIN`                                   | 두 테이블을 어떻게 잇습니까?                   | 45분                 |
| **3**         | `LEFT JOIN` · `RIGHT JOIN`                     | 짝이 없는 행도 보려면?                         | 40분                 |
| **4**         | `UNION` · `UNION ALL`                          | 결과를 세로로 이어 붙이려면?                   | 25분                 |
| **5**         | 서브쿼리 — `WHERE`절 · `FROM`절 · 스칼라       | 쿼리 안에 쿼리를 넣으려면?                     | 40분                 |
| **6**         | 세 테이블 이상 결합과 `EXISTS`                 | 테이블이 셋 이상이면?                          | 40분                 |
| **연습 문제** | 조인과 서브쿼리 5문항                          | 조인·서브쿼리를 반복 훈련합니다                | 30분                 |
| **종합 실습** | 월별·카테고리별 매출 TOP 10                    | 세 테이블을 이어 매출 순위를 뽑을 수 있습니까? | 40분                 |
| **정리**      | 오늘의 퀴즈 · 개념 연결 도식 · 치트시트 · 과제 | 오늘 배운 것을 확인하고 정리합니다             | 20분                 |
|               | _합계_                                         |                                                | _320분 (5시간 20분)_ |

_⏱️ 학습 속도는 사람마다 다릅니다 — 여러분의 속도로 가면 됩니다._

# Part 0. 오늘의 여정

## 🔁 지난 시간 복습

이전 시간에는 _한 테이블 안에서_ 데이터를 다뤘습니다.

- `ORDER BY`·`LIMIT` — 정렬하고 상위 N개 보기
- `DISTINCT`·`AS`·`LIKE` — 중복 제거·별칭·패턴 검색
- `CASE WHEN` — 조건으로 분류
- 집계함수·`GROUP BY`·`HAVING` — 그룹별 요약

그런데 한계가 있었습니다. `orders`에는 *고객 이름*도, *상품 카테고리*도 없습니다. 그 정보는 `customers`·`products`라는 **다른 테이블**에 있습니다. 오늘은 이 흩어진 테이블들을 다시 **잇습니다**.

> ❓ **질문하기**  
> 데이터는 왜 여러 테이블로 나뉘어 있고, 어떻게 다시 합칠까요?

## ⚙️ 환경 설정과 실습 데이터

> ⚠️ **주의하기 — 셀은 위에서부터 순서대로 실행**  
> 코드 셀은 반드시 위에서부터 순서대로 실행합니다. 중간부터 실행하면 이전 셀의 변수를 찾지 못해 오류가 납니다.

이전과 동일한 `customers`·`orders`·`products`에 더해, 오늘은 **주문 상세(`order_items`)** 테이블을 추가합니다. 한 주문에 어떤 상품이 몇 개 담겼는지를 담은 표입니다.

## 실습 환경 — LMS 또는 BigQuery

이 노트북의 SQL은 **LMS 실습 환경(DuckDB)** 또는 **BigQuery** 어느 쪽에서든 실행되도록 작성했습니다. 둘 중 **편한 쪽 하나만** 고르면 됩니다.

- **LMS에서 학습한다면** — 바로 아래 `[C1]` 준비 셀을 먼저 실행한 뒤, 이어지는 셀을 위에서부터 순서대로 실행합니다.
- **BigQuery에서 학습한다면** — `[C1]`은 건너뛰고, 각 셀의 SQL을 BigQuery 콘솔(또는 샌드박스)에 붙여 실행합니다. 테이블 이름의 `project_name.dataset_name` 부분만 여러분의 프로젝트·데이터셋 이름으로 바꾸면 됩니다.

> ⚠️ **주의하기 — 두 환경의 문법이 다른 곳은 딱 두 군데(날짜 함수)뿐입니다**  
> 해당 셀에는 `/* BigQuery: ... */` 형태로 두 표기를 나란히 적어 두었으니, 자기 환경에 맞는 쪽을 쓰면 됩니다. 전체 대조는 아래 표에서 확인합니다.

## 두 환경의 문법 차이 — 한눈에 보기

| 항목              | LMS (DuckDB)                   | BigQuery                   | 이 교안에서는                                                   |
| ----------------- | ------------------------------ | -------------------------- | --------------------------------------------------------------- |
| 테이블 이름       | `project_name.dataset_name.표` | 좌동                       | **공통 사용** — `[C1]` 준비 셀이 같은 이름 공간을 만들어 줍니다 |
| 문자열 타입       | `STRING`                       | `STRING`                   | 공통                                                            |
| 정수 타입         | `INT64`                        | `INT64`                    | 공통                                                            |
| 금액 타입         | `DECIMAL(12,2)`                | `DECIMAL(12,2)`            | 공통 (BigQuery는 `NUMERIC`으로 해석)                            |
| 합집합(중복 제거) | `UNION DISTINCT`               | `UNION DISTINCT`           | 공통 표기로 통일                                                |
| **월 단위 절단**  | `date_trunc('month', d)`       | `DATE_TRUNC(d, MONTH)`     | **함수명·인자 순서가 다름 → 병기**                              |
| **날짜 차이**     | `date_diff('day', 시작, 끝)`   | `DATE_DIFF(끝, 시작, DAY)` | **인자 순서가 반대 → 병기**                                     |

> 💡 **핵심짚기 — 나머지 문법은 전부 같습니다**  
> `SELECT`·`WHERE`·`GROUP BY`·`HAVING`·`JOIN`·`UNION`·서브쿼리·`CASE WHEN`·`COALESCE`·`NULLIF`·윈도우 함수·`ROLLUP`·`CREATE TABLE AS`까지, 이 모듈에서 배우는 거의 모든 문법이 두 환경에서 똑같이 동작합니다. **"SQL은 표준 언어"** 라는 말이 실감 나는 대목입니다.

> 📌 **실무 연결하기 — 왜 날짜 함수만 다를까요**  
> 날짜 처리는 SQL 표준이 늦게 정리된 영역이라 DB마다 독자 함수를 먼저 만들었습니다. 그래서 실무에서 DB를 옮길 때 **가장 먼저 깨지는 것도 날짜 함수**입니다. 지금 이 차이를 눈에 익혀 두면 나중에 크게 도움이 됩니다.

In [ ]:
-- [C1] LMS(DuckDB) 준비 — BigQuery로 학습한다면 이 셀은 건너뛰세요.
-- BigQuery와 똑같은 `project_name.dataset_name.표` 이름을 쓸 수 있도록 이름 공간을 만듭니다.
ATTACH IF NOT EXISTS ':memory:' AS project_name;
CREATE SCHEMA IF NOT EXISTS project_name.dataset_name;

In [ ]:
-- [C2]
CREATE OR REPLACE TABLE project_name.dataset_name.customers (
  customer_id STRING,
  name STRING,
  country STRING,
  signup_date DATE,
  grade STRING
);

INSERT INTO project_name.dataset_name.customers VALUES
  ('C001', '김민준', 'Korea', '2023-01-05', 'Gold'),
  ('C002', '이서연', 'Korea', '2023-02-11', 'Silver'),
  ('C003', '박도윤', 'Japan', '2023-02-20', 'Bronze'),
  ('C004', '최지우', 'USA', '2023-03-03', 'Gold'),
  ('C005', '정하준', 'Korea', '2023-03-15', 'Silver'),
  ('C006', '강서윤', 'Korea', '2023-04-01', 'Bronze'),
  ('C007', '조은우', 'Japan', '2023-04-18', 'Silver'),
  ('C008', '윤지호', 'USA', '2023-05-09', 'Gold'),
  ('C009', '임하은', 'Korea', '2023-05-22', 'Bronze'),
  ('C010', '한예준', 'Korea', '2023-06-02', 'Silver'),
  ('C011', '오시우', NULL, '2023-06-19', 'Bronze'),
  ('C012', '신아린', 'Japan', '2023-07-07', 'Silver'),
  ('C013', '권준서', 'Korea', '2023-07-25', 'Gold'),
  ('C014', '황지안', 'USA', '2023-08-10', NULL),
  ('C015', '안수아', 'Korea', '2023-08-28', 'Bronze')

In [ ]:
-- [C3]
CREATE OR REPLACE TABLE project_name.dataset_name.products (
  product_id STRING,
  product_name STRING,
  category STRING,
  price DECIMAL(12,2)
);

INSERT INTO project_name.dataset_name.products VALUES
  ('P01', '에어러너', 'Running', 89000),
  ('P02', '클래식 스니커즈', 'Sneakers', 65000),
  ('P03', '첼시 부츠', 'Boots', 145000),
  ('P04', '여름 샌들', 'Sandals', 38000),
  ('P05', '트레일 러너', 'Running', 119000),
  ('P06', '캔버스 스니커즈', 'Sneakers', 49000),
  ('P07', '워커 부츠', 'Boots', 175000),
  ('P08', '슬리퍼 샌들', 'Sandals', 25000),
  ('P09', '양말 세트', 'Accessory', 12000),
  ('P10', '운동화 끈', 'Accessory', 5000)

In [ ]:
-- [C4]
CREATE OR REPLACE TABLE project_name.dataset_name.orders (
  order_id STRING,
  customer_id STRING,
  order_date DATE,
  status STRING,
  amount DECIMAL(12,2)
);

INSERT INTO project_name.dataset_name.orders VALUES
  ('O0001', 'C001', '2023-09-02', 'Paid', 125000),
  ('O0002', 'C002', '2023-09-05', 'Shipped', 89000),
  ('O0003', 'C001', '2023-09-11', 'Returned', 45000),
  ('O0004', 'C003', '2023-09-15', 'Paid', 230000),
  ('O0005', 'C004', '2023-09-20', 'Cancelled', NULL),
  ('O0006', 'C005', '2023-09-25', 'Shipped', 67000),
  ('O0007', 'C002', '2023-10-01', 'Paid', 158000),
  ('O0008', 'C006', '2023-10-04', 'Placed', 32000),
  ('O0009', 'C007', '2023-10-12', 'Shipped', 410000),
  ('O0010', 'C008', '2023-10-19', 'Paid', 99000),
  ('O0011', 'C001', '2023-10-23', 'Paid', 76000),
  ('O0012', 'C009', '2023-10-28', 'Cancelled', NULL),
  ('O0013', 'C010', '2023-11-02', 'Shipped', 142000),
  ('O0014', 'C004', '2023-11-08', 'Paid', 88000),
  ('O0015', 'C011', '2023-11-13', 'Placed', 53000),
  ('O0016', 'C012', '2023-11-19', 'Shipped', 175000),
  ('O0017', 'C002', '2023-11-24', 'Returned', 61000),
  ('O0018', 'C013', '2023-11-29', 'Paid', 320000),
  ('O0019', 'C005', '2023-12-03', 'Paid', 47000),
  ('O0020', 'C008', '2023-12-09', 'Shipped', 215000),
  ('O0021', 'C014', '2023-12-14', 'Placed', 38000),
  ('O0022', 'C001', '2023-12-20', 'Paid', 134000),
  ('O0023', 'C015', '2023-12-25', 'Shipped', 92000),
  ('O0024', 'C007', '2024-01-03', 'Paid', 268000),
  ('O0025', 'C010', '2024-01-09', 'Cancelled', NULL),
  ('O0026', 'C003', '2024-01-15', 'Paid', 119000),
  ('O0027', 'C013', '2024-01-22', 'Shipped', 405000),
  ('O0028', 'C006', '2024-01-28', 'Paid', 58000),
  ('O0029', 'C004', '2024-02-04', 'Returned', 73000),
  ('O0030', 'C012', '2024-02-11', 'Paid', 187000)

In [ ]:
-- [C5]
CREATE OR REPLACE TABLE project_name.dataset_name.order_items (
  order_id STRING,
  product_id STRING,
  quantity INT64,
  unit_price DECIMAL(12,2)
);

INSERT INTO project_name.dataset_name.order_items VALUES
  ('O0001', 'P01', 1, 125000),
  ('O0002', 'P02', 2, 44500),
  ('O0003', 'P03', 1, 45000),
  ('O0004', 'P04', 2, 115000),
  ('O0006', 'P05', 1, 67000),
  ('O0007', 'P06', 2, 79000),
  ('O0008', 'P07', 1, 32000),
  ('O0009', 'P08', 2, 205000),
  ('O0010', 'P09', 1, 99000),
  ('O0011', 'P01', 2, 38000),
  ('O0013', 'P02', 1, 142000),
  ('O0014', 'P03', 2, 44000),
  ('O0015', 'P04', 1, 53000),
  ('O0016', 'P05', 2, 87500),
  ('O0017', 'P06', 1, 61000),
  ('O0018', 'P07', 2, 160000),
  ('O0019', 'P08', 1, 47000),
  ('O0020', 'P09', 2, 107500),
  ('O0021', 'P01', 1, 38000),
  ('O0022', 'P02', 2, 67000),
  ('O0023', 'P03', 1, 92000),
  ('O0024', 'P04', 2, 134000),
  ('O0026', 'P05', 1, 119000),
  ('O0027', 'P06', 2, 202500),
  ('O0028', 'P07', 1, 58000),
  ('O0029', 'P08', 2, 36500),
  ('O0030', 'P09', 1, 187000)

> 오늘 쓰는 네 테이블의 관계를 그림으로 보면 이렇습니다.

```
customers (고객)            products (상품)
   │ customer_id               │ product_id
   │ (PK)                       │ (PK)
   ▼                            ▼
orders (주문) ──order_id(PK)── order_items (주문상세)
   customer_id(FK)               order_id(FK), product_id(FK)
```

> 화살표는 "번호로 연결된다"는 뜻입니다. `orders.customer_id`는 어떤 고객인지, `order_items`의 `order_id`·`product_id`는 어떤 주문의 어떤 상품인지를 가리킵니다.

# Part 1. 기본키(PK)와 외래키(FK)

테이블을 합치려면, 먼저 _무엇을 기준으로_ 잇는지 알아야 합니다. 그 기준이 되는 열이 **기본키(PK)** 와 **외래키(FK)** 입니다.

> ❓ **질문하기**  
> 테이블끼리는 무엇을 기준으로 연결되어 있습니까?

## 🧩 개념 잡기

- **기본키(Primary Key, PK)** = 각 행을 **유일하게 구분하는 열 또는 열 조합**입니다. 값이 중복되거나 `NULL`이면 안 됩니다. `customers`의 `customer_id`, `orders`의 `order_id`가 이 역할을 합니다.
- **외래키(Foreign Key, FK)** = _다른 표의 기본키를 가리키는_ 열입니다. `orders`의 `customer_id`는 "이 주문이 어느 고객 것인지"를 `customers`를 가리켜 알려줍니다.

```
customers                 orders
 customer_id(PK)           order_id(PK)
 C001  김민준    ◄───────  customer_id(FK) = C001
                           "이 주문은 C001(김민준)의 것"
```

> ⚠️ **주의하기 — 키 역할과 키 제약은 다릅니다**  
> 현재 실습 테이블의 생성문에는 `PRIMARY KEY`·`FOREIGN KEY` 제약을 직접 선언하지 않았습니다. 따라서 이 열들은 데이터 모델에서 PK·FK **역할**을 하지만 DBMS가 중복·누락·잘못된 참조를 자동으로 막아 주지는 않습니다. 제약이 선언되어야 앞으로 입력될 데이터까지 규칙을 보장하며, 조회로 확인한 유일성은 현재 데이터 상태만 보여줍니다.

## 왜 이렇게 나눕니까?

만약 주문마다 고객 이름·국가를 통째로 적으면, 김민준이 4번 주문할 때 '김민준/Korea'가 4번 중복됩니다. 대신 `orders`에는 **`customer_id`(FK)만** 적고, 이름·국가는 `customers`에 _한 번만_ 둡니다. 이렇게 하면:

- 중복이 줄고
- 고객 이름이 바뀌어도 `customers` 한 곳만 고치면 되며
- 필요할 때 FK로 다시 이어 붙일 수 있습니다 ← 이게 바로 오늘 배울 `JOIN`!

> 📌 **실무 연결하기 — 관계형 DB의 뼈대**  
> PK/FK는 모든 관계형 DB의 뼈대입니다 — 병원은 _환자번호(PK)_ 로 진료기록을, 은행은 _계좌번호(PK)_ 로 거래내역을, 학교는 _학번(PK)_ 으로 수강기록을 잇습니다.

> 🐼 **pandas와 비교**  
> PK/FK는 pandas에서 `merge`의 _기준 열(key)_ 과 같은 역할입니다. `pd.merge(orders, customers, on='customer_id')` 에서 `on='customer_id'` 가 바로 두 표를 잇는 열입니다. SQL은 이 연결을 `ON o.customer_id = c.customer_id`로 명시합니다.

## ⌨️ 백문이 불여일타 (1)

```
[문제] 현재 데이터에서 customer_id가 기본키 후보 자격(유일하고 NULL이 없음)을 갖추었는지 확인합니다.
전체 행 수(COUNT(*))와 고유 customer_id 수(COUNT(DISTINCT customer_id))가 같으면 중복이 없다는 뜻입니다.
```

In [ ]:
-- [C6] ⌨️ 백문이 불여일타 (1) — customer_id의 PK 자격(유일성) 확인

-- 여기에 SQL 쿼리를 작성하세요.

SELECT
  count(*) AS 전체행,
  count(DISTINCT customer_id) AS 고유고객
FROM project_name.dataset_name.customers;


<details>
<summary>(펼치기) 🔑 힌트</summary>

- 두 값을 한 `SELECT` 안에서 나란히 구하면 한 번에 비교할 수 있습니다.
- 전체 행 수는 `COUNT(*)`, 고유 고객 수는 `COUNT(DISTINCT customer_id)`로 셉니다.
- 두 값이 같으면 현재 데이터에는 중복과 `NULL`이 없으므로 PK 후보 자격이 있습니다. 다만 조회 결과가 앞으로 입력될 값까지 보장하지는 않습니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  COUNT(*) AS 전체행,
  COUNT(DISTINCT customer_id) AS 고유고객
FROM project_name.dataset_name.customers
```

</details>

## 🚦 짚고 넘어가기

다음 질문에 답할 수 있으면 다음 Part로 넘어가세요. 틀려도 괜찮습니다.

1. 기본키(PK)는 무엇을 보장합니까?
2. `orders.customer_id`는 PK입니까, FK입니까?
3. 테이블을 나눠 담으면 어떤 장점이 있습니까?

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

1. 각 행을 유일하게 구분할 수 있음을 보장합니다. 값이 중복되거나 비어(`NULL`) 있으면 안 됩니다.
2. FK(외래키)입니다. `customers` 테이블의 PK인 `customer_id`를 가리킵니다. (`orders` 자신의 PK는 `order_id`입니다.)
3. 같은 정보가 여러 곳에 중복 저장되는 것을 막고, 수정할 때 한 곳만 고치면 되게 해줍니다.

</details>

> ⏭️ **다음 학습 예고**  
> 연결 열(PK·FK)을 알았으니, 이제 이 열을 기준으로 두 테이블을 **합치는** `JOIN`을 배웁니다.

# Part 2. `INNER JOIN`

`JOIN`은 두 테이블을 **연결 열(PK=FK)** 을 기준으로 옆으로 나란히 붙입니다. 가장 기본은 양쪽에 _모두 짝이 있는_ 행만 합치는 `INNER JOIN`입니다.

> ❓ **질문하기**  
> 두 표의 정보를 어떻게 한 줄로 합칩니까?

## 🧩 개념 잡기

`JOIN`은 **같은 번호끼리 카드를 맞추는 것** 입니다. 주문 카드의 `customer_id`와 고객 카드의 `customer_id`가 같으면, 두 카드를 옆으로 붙입니다.

```
orders                 customers
 O0001 │ C001    +    C001 │ 김민준   →   O0001 │ C001 │ 김민준 │ ...
```

> 🐼 **pandas와 비교**  
> `JOIN ... ON a.key = b.key` ↔ `pd.merge(a, b, on='key')` — 같은 일입니다!

## `INNER JOIN` 문법

```sql
SELECT
  열들
FROM 테이블A AS a
JOIN 테이블B AS b
  ON a.연결열 = b.연결열
```

- `AS a`, `AS b` 는 테이블에 짧은 **별칭**을 주는 것(긴 이름 대신 `a.`, `b.`로 참조).
- `ON` 뒤에 **연결 조건**(보통 PK = FK)을 씁니다.
- 그냥 `JOIN`은 `INNER JOIN`과 같습니다(양쪽에 짝이 있는 행만).

▶️ **코드 실행하기 · 코드 셀 7 [C7]**  
— **금액이 있는 주문 중 금액이 큰 상위 5건**을 뽑아, 각 주문에 고객 이름과 국가를 붙입니다.

In [ ]:
-- [C7]
SELECT
  o.order_id,
  o.amount,
  c.name AS 고객명,
  c.country AS 국가
FROM project_name.dataset_name.orders AS o
JOIN project_name.dataset_name.customers AS c
  ON o.customer_id = c.customer_id
WHERE
  o.amount IS NOT NULL
ORDER BY
  o.amount DESC
LIMIT 5

> 👀 **결과읽기 — [C7]**  
> `orders`에는 없던 *고객 이름·국가*가 `customers`에서 붙어 왔습니다. `o.`·`c.`는 어느 테이블의 열인지 구분하는 표시입니다.

▶️ **코드 실행하기 · 코드 셀 8 [C8]**  
— `order_items`에 `products`를 이어 상품명·카테고리를 붙이고, **수량 × 단가로 라인별 매출을 계산해 큰 순으로 상위 5건**을 조회합니다.

In [ ]:
-- [C8]
SELECT
  oi.order_id,
  p.product_name AS 상품명,
  p.category AS 카테고리,
  oi.quantity AS 수량,
  oi.unit_price AS 단가,
  oi.quantity * oi.unit_price AS 라인합계
FROM project_name.dataset_name.order_items AS oi
JOIN project_name.dataset_name.products AS p
  ON oi.product_id = p.product_id
ORDER BY
  라인합계 DESC
LIMIT 5

> 👀 **결과읽기 — [C8]**  
> `order_items`(수량·단가)에 `products`(상품명·카테고리)를 붙여, 주문 상세를 사람이 읽을 수 있게 만들었습니다. `수량 × 단가`로 라인별 매출도 바로 계산했습니다.

> 📌 **실무 연결하기 — 분리된 사실을 하나의 뷰로**  
> `JOIN`은 _주문↔고객_(이커머스), _거래↔계좌_(금융), _진료↔환자_(헬스케어)처럼 분리된 사실을 하나의 분석 뷰로 모읍니다.

## ⌨️ 백문이 불여일타 (2)

```
[문제] orders와 customers를 합쳐, 각 주문의 주문번호·금액·고객 등급(grade)을 조회합니다.
(금액 있는 주문 중 금액이 큰 상위 5건)
```

In [ ]:
-- [C9] ⌨️ 백문이 불여일타 (2) — 주문번호·금액·고객 등급 조회

-- 여기에 SQL 쿼리를 작성하세요.

SELECT
  O.order_id,
  O.amount,
  C.grade
FROM project_name.dataset_name.orders AS O
JOIN project_name.dataset_name.customers AS C
  ON O.customer_id = C.customer_id
WHERE amount IS NOT NULL
ORDER BY O.amount DESC, O.order_id
LIMIT 5


<details>
<summary>(펼치기) 🔑 힌트</summary>

- `FROM ... orders AS o`, `JOIN ... customers AS c`처럼 별칭을 붙이면 열 구분이 쉬워집니다.
- 두 테이블은 `ON o.customer_id = c.customer_id`로 잇습니다.
- 금액 있는 것만 남기려면 `WHERE o.amount IS NOT NULL`을 사용합니다.
- 금액 상위 5건은 `ORDER BY o.amount DESC, o.order_id`로 순서를 정한 뒤 `LIMIT 5`를 사용합니다. 두 번째 기준은 금액이 같을 때도 결과 순서를 일정하게 만듭니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  o.order_id,
  o.amount,
  c.grade
FROM project_name.dataset_name.orders AS o
JOIN project_name.dataset_name.customers AS c
  ON o.customer_id = c.customer_id
WHERE
  o.amount IS NOT NULL
ORDER BY
  o.amount DESC,
  o.order_id
LIMIT 5
```

</details>

## ⌨️ 백문이 불여일타 (3)

```
[문제] order_items와 products를 합쳐, 카테고리별 총 판매수량을 구합니다. (수량 합이 많은 순)
```

In [ ]:
-- [C10] ⌨️ 백문이 불여일타 (3) — 카테고리별 총 판매수량

-- 여기에 SQL 쿼리를 작성하세요.

SELECT p.category, SUM(oi.quantity) AS 총수량 
FROM project_name.dataset_name.order_items AS oi
JOIN project_name.dataset_name.products AS p
 ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY 총수량 DESC

<details>
<summary>(펼치기) 🔑 힌트</summary>

- `ON oi.product_id = p.product_id`로 `JOIN`해 상품 정보(카테고리)를 붙입니다.
- `GROUP BY p.category`로 묶고, 수량 합은 `SUM(oi.quantity)`로 구합니다.
- 많은 순 정렬은 `ORDER BY 총수량 DESC`처럼 별칭으로 하면 편합니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  p.category,
  SUM(oi.quantity) AS 총수량
FROM project_name.dataset_name.order_items AS oi
JOIN project_name.dataset_name.products AS p
  ON oi.product_id = p.product_id
GROUP BY
  p.category
ORDER BY
  총수량 DESC
```

</details>

> ⚠️ **주의하기 — 관계의 종류에 따라 결과 행 수가 달라집니다**  
> 1:1 관계는 한 행이 한 행과, 1:N 관계는 왼쪽 한 행이 오른쪽 여러 행과 연결됩니다. `customers`→`orders`와 `orders`→`order_items`는 1:N 관계입니다. 한 주문에 상품이 여러 개 담기면 `orders`와 `order_items`를 `JOIN`할 때 주문 한 건이 상품 수만큼 늘어납니다. 이는 오류가 아니지만, 이 상태에서 `SUM(o.amount)`를 하면 주문 금액이 중복으로 더해질 수 있습니다. 양쪽 연결 열이 모두 중복된 N:M 관계라면 행 수는 더 크게 늘어날 수 있습니다.

```
orders            order_items              JOIN 결과
 O0002 89000  +   O0002 P02 2개   →   O0002 ... 89000 ... P02 (2줄로 늘 수 있음)
```

▶️ **코드 실행하기 · 코드 셀 11 [C11]**  
— `orders`와 `order_items`를 이어, **주문 한 건이 담긴 상품 수만큼 늘어나는 것**을 6행으로 확인합니다.

In [ ]:
-- [C11]
SELECT
  o.order_id,
  o.amount,
  oi.product_id,
  oi.quantity
FROM project_name.dataset_name.orders AS o
JOIN project_name.dataset_name.order_items AS oi
  ON o.order_id = oi.order_id
ORDER BY
  o.order_id NULLS LAST
LIMIT 6

> 👀 **결과읽기 — [C11]**  
> `JOIN` 후 결과 한 행은 주문 한 건이 아니라 **주문 상품 한 줄**을 뜻합니다. 그래서 "주문 매출 합"을 구할 때는 `orders`만으로 `SUM(amount)`하거나, 라인 매출은 `SUM(quantity * unit_price)`로 구하는 식으로 무엇을 세는지 분명히 해야 합니다. 조인 전후의 `COUNT(*)`와 `COUNT(DISTINCT o.order_id)`를 함께 비교하면 행 증가가 의도한 결과인지 확인할 수 있습니다.

## ⌨️ 백문이 불여일타 (4)

```
[문제] 국가별 매출을 구합니다.
1) orders와 customers를 JOIN 합니다.
2) country로 묶어 매출 합(SUM(amount))을 구합니다. (금액 있는 것만)
3) 국가 정보가 없는(NULL) 고객은 제외합니다.
4) 매출 큰 순으로 정렬합니다.
```

In [ ]:
-- [C12] ⌨️ 백문이 불여일타 (4) — 국가별 매출 구하기

-- 여기에 SQL 쿼리를 작성하세요.

SELECT
  c.country,
  SUM(o.amount) AS 매출
FROM project_name.dataset_name.orders AS o
JOIN project_name.dataset_name.customers As c
  ON o.customer_Id = c.customer_id
WHERE o.amount IS NOT NULL AND c.country IS NOT NULL
GROUP BY c.country
ORDER BY 매출 DESC

<details>
<summary>(펼치기) 🔑 힌트</summary>

- `orders`와 `customers`를 `ON o.customer_id = c.customer_id`로 `JOIN`합니다.
- `WHERE`에서 `o.amount IS NOT NULL AND c.country IS NOT NULL`을 함께 적용합니다.
- `GROUP BY c.country`로 묶어 `SUM(o.amount)`을 구하고, `ORDER BY 매출 DESC`로 정렬합니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  c.country,
  SUM(o.amount) AS 매출
FROM project_name.dataset_name.orders AS o
JOIN project_name.dataset_name.customers AS c
  ON o.customer_id = c.customer_id
WHERE
  o.amount IS NOT NULL AND c.country IS NOT NULL
GROUP BY
  c.country
ORDER BY
  매출 DESC
```

</details>

## 🚦 짚고 넘어가기

1. `JOIN ... ON` 의 `ON` 뒤에는 보통 무엇이 옵니까?
2. 테이블 별칭(`AS o`, `AS c`)은 왜 씁니까?
3. `INNER JOIN`은 어떤 행을 결과에 포함합니까?
4. `orders`와 `order_items`를 조인한 결과 한 행은 무엇을 뜻하며, 주문 수는 어떤 식으로 세어야 합니까?

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

1. 두 테이블을 잇는 연결 조건이 옵니다. 보통 `FK = PK` 형태(예: `o.customer_id = c.customer_id`)입니다.
2. 긴 테이블 이름을 짧게 줄이고, 같은 이름의 열이 어느 테이블 것인지(`o.`·`c.`) 구분하기 위해서입니다.
3. 양쪽 테이블에 모두 짝이 있는 행만 결과에 포함합니다.
4. 결과 한 행은 주문 상품 한 줄을 뜻합니다. 주문 수는 `COUNT(*)`가 아니라 `COUNT(DISTINCT o.order_id)`로 세어야 중복을 피할 수 있습니다.

</details>

> ⏭️ **다음 학습 예고**  
> `INNER JOIN`은 _양쪽에 짝이 있는_ 행만 남깁니다. 그런데 "주문이 한 번도 없는 상품"처럼 _짝이 없는_ 행도 보고 싶다면? `LEFT JOIN`이 필요합니다.

# Part 3. `LEFT JOIN` · `RIGHT JOIN`

`INNER JOIN`은 양쪽에 짝이 있어야만 결과에 남습니다. 하지만 _"한 번도 안 팔린 상품"_, _"상세가 없는 주문"_ 처럼 **짝이 없는 행**도 분석에선 중요합니다. 이때 `LEFT JOIN`을 씁니다.

> ❓ **질문하기**  
> 짝이 없는 행은 어떻게 결과에 살려둡니까?

## 🧩 개념 잡기

- `INNER JOIN` = **교집합** (양쪽에 다 있는 것만)
- `LEFT JOIN` = **왼쪽 테이블은 전부 + 오른쪽은 짝 있으면 붙이고, 없으면 `NULL`**

```
products (LEFT, 전부 유지)      order_items (있으면 붙임)
 P01 에어러너    ─ 팔림 →  수량 O
 P10 운동화 끈   ─ 안 팔림 → 수량 NULL  ← INNER였다면 사라졌을 행
```

`RIGHT JOIN`은 방향만 반대(오른쪽 테이블 전부 유지)입니다.

> ⚠️ **주의하기 — 오른쪽 열의 조건을 `WHERE`에 쓰면 행이 사라질 수 있습니다**  
> `LEFT JOIN` 뒤에 `WHERE oi.quantity >= 1`처럼 오른쪽 테이블의 일반 조건을 쓰면 짝이 없어 `NULL`이 된 행이 제외되어 `INNER JOIN`과 비슷한 결과가 됩니다. 왼쪽 행을 모두 유지하면서 오른쪽 행만 제한하려면 조건을 `ON`에 함께 작성합니다. 단, `WHERE oi.order_id IS NULL`은 짝이 없는 행만 찾으려는 의도적인 예외입니다.

▶️ **코드 실행하기 · 코드 셀 13 [C13]**  
— `products`를 기준으로 `LEFT JOIN`해, **한 번도 팔리지 않은 상품까지 남긴** 상품별 총판매수량을 구합니다.

In [ ]:
-- [C13]
SELECT
  p.product_id,
  p.product_name AS 상품명,
  SUM(oi.quantity) AS 총판매수량
FROM project_name.dataset_name.products AS p
LEFT JOIN project_name.dataset_name.order_items AS oi
  ON p.product_id = oi.product_id
GROUP BY
  p.product_id,
  상품명
ORDER BY
  총판매수량 NULLS LAST

> 👀 **결과읽기 — [C13]**  
> 결과 맨 위에 `총판매수량`이 비어 있는(`NaN`/`NULL`) 상품이 보입니까? 한 번도 팔리지 않은 상품입니다. `INNER JOIN`이었다면 아예 사라졌을 행입니다. `LEFT JOIN` 덕분에 "안 팔린 상품"을 발견할 수 있습니다.

> 📌 **실무 연결하기 — '비어 있는 쪽' 찾기**  
> `LEFT JOIN`은 _주문 없는 신규 회원_(이커머스), _거래 없는 휴면 계좌_(금융), _방문 없는 환자_(헬스케어)처럼 **"비어 있는 쪽"** 을 찾을 때 결정적입니다.

## ⌨️ 백문이 불여일타 (5)

```
[문제] 위 결과에서 한 걸음 더 나아가, 한 번도 팔리지 않은 상품만 보여주세요.
```

In [ ]:
-- [C14] ⌨️ 백문이 불여일타 (5) — 한 번도 팔리지 않은 상품 찾기

-- 여기에 SQL 쿼리를 작성하세요.

SELECT
  p.product_id,
  p.product_name 
FROM project_name.dataset_name.products AS p
LEFT JOIN project_name.dataset_name.order_items AS oi
  ON p.product_id = oi.product_id
WHERE oi.order_id IS NULL

<details>
<summary>(펼치기) 🔑 힌트</summary>

- `products`를 왼쪽에 두고 `LEFT JOIN order_items`를 하면 안 팔린 상품도 결과에 남습니다.
- 짝이 없는 행은 `order_items` 쪽 열이 `NULL`이 됩니다.
- `WHERE oi.order_id IS NULL` 조건으로 그런 행만 남깁니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  p.product_id,
  p.product_name
FROM project_name.dataset_name.products AS p
LEFT JOIN project_name.dataset_name.order_items AS oi
  ON p.product_id = oi.product_id
WHERE
  oi.order_id IS NULL
```

</details>

## `RIGHT JOIN`은 방향만 반대

`LEFT JOIN`이 _왼쪽_ 테이블을 전부 남긴다면, `RIGHT JOIN`은 _오른쪽_ 테이블을 전부 남깁니다. 사실 `A LEFT JOIN B`와 `B RIGHT JOIN A`는 같은 결과입니다. 그래서 실무에서는 대부분 **`LEFT JOIN`만** 쓰고 필요하면 테이블 순서를 바꿉니다.

```
A LEFT  JOIN B   =   B RIGHT JOIN A      (왼쪽 전부 유지)
```

> 🐼 **pandas와 비교**  
> `LEFT JOIN` ↔ `pd.merge(a, b, on='key', how='left')`, `RIGHT JOIN` ↔ `how='right'`, `INNER` ↔ `how='inner'`. pandas의 `how` 옵션이 곧 `JOIN` 종류입니다.

▶️ **코드 실행하기 · 코드 셀 15 [C15]**  
— 같은 결과를 **`RIGHT JOIN`으로** 구해, 기준 테이블만 바꾸면 `LEFT JOIN`과 같아지는 것을 확인합니다.

In [ ]:
-- [C15]
SELECT
  p.product_name AS 상품명,
  SUM(oi.quantity) AS 총판매수량
FROM project_name.dataset_name.order_items AS oi
RIGHT JOIN project_name.dataset_name.products AS p
  ON oi.product_id = p.product_id
GROUP BY
  상품명
ORDER BY
  총판매수량 NULLS LAST

> 👀 **결과읽기 — [C15]**  
> `order_items RIGHT JOIN products`는 _오른쪽(products)을 전부_ 남기므로, 안 팔린 상품도 결과에 남습니다(`products LEFT JOIN order_items`와 같은 결과입니다).

## 🚦 짚고 넘어가기

1. `INNER JOIN`과 `LEFT JOIN`의 차이를 한 문장으로 말합니다.
2. `LEFT JOIN`에서 오른쪽에 짝이 없으면 그 열들은 어떤 값이 됩니까?
3. "짝이 없는 행만" 찾으려면 어떤 조건을 추가합니까?

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

1. `INNER JOIN`은 양쪽에 짝이 있는 행만 남기고, `LEFT JOIN`은 짝이 없어도 왼쪽 테이블의 행을 전부 남깁니다.
2. 그 열들은 `NULL`이 됩니다.
3. 오른쪽 테이블의 열이 비어 있는지 확인하는 조건(예: `WHERE oi.order_id IS NULL`)을 추가합니다.

</details>

> ⏭️ **다음 학습 예고**  
> `JOIN`은 테이블을 _옆으로_ 붙였습니다. 그런데 두 결과 목록을 _세로로_ 이어 붙이고 싶다면? `UNION`을 배웁니다.

# Part 4. `UNION` · `UNION ALL`

`JOIN`이 테이블을 _옆으로_(열 추가) 붙였다면, `UNION`은 두 결과를 _세로로_(행 추가) 이어 붙입니다. "Gold 고객 목록"과 "큰손 고객 목록"을 하나의 명단으로 합치는 것과 같습니다.

> ❓ **질문하기**  
> 두 개의 결과 목록을 하나로 어떻게 합칩니까?

## 🧩 개념 잡기

```
[목록 A]        [목록 B]              [UNION 결과]
 김민준          최지우         →       김민준
 권준서          김민준                 권준서
                                        최지우   (중복 김민준은 1번만)
```

| 구분        | 동작                                      |
| ----------- | ----------------------------------------- |
| `UNION`     | 두 결과를 합치고 **중복 제거**            |
| `UNION ALL` | 두 결과를 합치되 **중복 그대로**(더 빠름) |

> ⚠️ **주의하기 — `UNION`은 열 개수·순서·타입이 같아야 한다**  
> 두 `SELECT`의 **열 개수와 순서·타입이 같아야** 합니다.

▶️ **코드 실행하기 · 코드 셀 16 [C16]**  
— **`Gold` 등급 명단과 한국 고객 명단을 위아래로 이어 붙여**, 사유 열과 함께 이름 순으로 조회합니다(`UNION ALL`).

In [ ]:
-- [C16]
SELECT
  name,
  '골드회원' AS 사유
FROM project_name.dataset_name.customers
WHERE
  grade = 'Gold'
UNION ALL
SELECT
  name,
  '한국고객' AS 사유
FROM project_name.dataset_name.customers
WHERE
  country = 'Korea'
ORDER BY
  name NULLS LAST

> 👀 **결과읽기 — [C16]**  
> 위는 `UNION ALL`이라 김민준처럼 두 조건(골드+한국)에 다 해당하면 _두 줄_ 나옵니다. 사유까지 보여주려고 일부러 `ALL`을 썼습니다. 순수 명단(중복 1번)만 원하면 `UNION`을 씁니다.

> 📌 **실무 연결하기 — 같은 형식의 데이터 모으기**  
> `UNION`은 _여러 채널의 거래를 한 장부로_(금융), _지점별 로그를 통합_(IT), _연도별 테이블 합치기_ 처럼 같은 형식의 데이터를 모을 때 쓰입니다.

## ⌨️ 백문이 불여일타 (6)

```
[문제] 금액이 30만원 이상인 주문의 주문번호와, 상태가 Returned인 주문의 주문번호를,
UNION으로 합쳐 하나의 '주의 주문' 목록을 만듭니다. (중복 제거)
```

In [ ]:
-- [C17] ⌨️ 백문이 불여일타 (6) — UNION으로 주의 주문 목록 만들기

-- 여기에 SQL 쿼리를 작성하세요.

SELECT
  order_id
FROM project_name.dataset_name.orders
WHERE
  amount >= 300000
UNION DISTINCT
SELECT
  order_id
FROM project_name.dataset_name.orders
WHERE
  status = 'Returned'

<details>
<summary>(펼치기) 🔑 힌트</summary>

- 조건별로 `SELECT order_id ...` 쿼리 두 개를 만들어 세로로 잇습니다.
- 두 `SELECT`는 열 개수·타입이 같아야 합니다. 여기서는 `order_id` 하나씩이면 됩니다.
- 중복 제거가 필요하므로 BigQuery는 `UNION DISTINCT`라고 명시해야 하므로 이 교안은 두 환경 공통으로 `UNION DISTINCT`를 씁니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  order_id
FROM project_name.dataset_name.orders
WHERE
  amount >= 300000
UNION DISTINCT
SELECT
  order_id
FROM project_name.dataset_name.orders
WHERE
  status = 'Returned'
```

</details>

▶️ **코드 실행하기 · 코드 셀 18 [C18]**  
— 같은 두 명단을 **`UNION DISTINCT`와 `UNION ALL`로 각각 합쳐 행 수를 비교**합니다.

In [ ]:
-- [C18]
SELECT
  'UNION DISTINCT (중복제거)' AS 방식,
  COUNT(*) AS 행수
FROM (
  SELECT
    name
  FROM project_name.dataset_name.customers
  WHERE
    grade = 'Gold'
  UNION DISTINCT
  SELECT
    name
  FROM project_name.dataset_name.customers
  WHERE
    country = 'Korea'
) AS t
UNION ALL
SELECT
  'UNION ALL (중복유지)',
  COUNT(*)
FROM (
  SELECT
    name
  FROM project_name.dataset_name.customers
  WHERE
    grade = 'Gold'
  UNION ALL
  SELECT
    name
  FROM project_name.dataset_name.customers
  WHERE
    country = 'Korea'
) AS t

> 👀 **결과읽기 — [C18]**  
> 같은 두 목록을 합쳐도 `UNION`(중복 제거)이 `UNION ALL`(중복 유지)보다 행 수가 적습니다(겹치는 사람이 1번만 남기 때문). 중복을 굳이 지울 필요가 없다면 _더 빠른_ `UNION ALL`을 쓰는 게 좋습니다.

## 🚦 짚고 넘어가기

1. `JOIN`과 `UNION`은 각각 어느 방향(옆/세로)으로 붙입니까?
2. `UNION`과 `UNION ALL`의 차이는?
3. `UNION`을 쓰려면 두 `SELECT`가 무엇을 맞춰야 합니까?

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

1. `JOIN`은 옆(가로)으로, `UNION`은 세로로 붙입니다.
2. `UNION DISTINCT`은 중복을 제거하고, `UNION ALL`은 중복을 그대로 두는 대신 더 빠릅니다.
3. 두 `SELECT`의 열 개수와 타입(순서 포함)을 맞춰야 합니다.

</details>

> ⏭️ **다음 학습 예고**  
> 마지막으로, "평균보다 비싼 주문", "미국 고객의 주문"처럼 **다른 쿼리의 결과를 조건으로** 쓰고 싶을 때가 있습니다. 쿼리 안에 쿼리를 넣는 서브쿼리를 배웁니다.

# Part 5. 서브쿼리 — `WHERE`절 · `FROM`절 · 스칼라

복잡한 질문은 한 번에 풀기 어렵습니다. _"평균 금액보다 비싼 주문"_ 은 ① 먼저 평균을 구하고 ② 그보다 큰 주문을 찾는 **두 단계**입니다. 이때 쿼리 안에 또 다른 쿼리(서브쿼리)를 넣습니다.

> ❓ **질문하기**  
> 복잡한 질문을 어떻게 단계로 나눠 풉니까?

## 🧩 개념 잡기

서브쿼리는 **바깥 쿼리가 사용할 중간 결과를 만드는 작은 쿼리**입니다. 논리적으로는 안쪽 결과를 구해야 바깥 조건을 판단할 수 있습니다. 실제 실행 과정에서는 DBMS의 최적화기가 서브쿼리를 조인 형태로 바꾸거나 실행 순서를 조정할 수 있습니다.

```sql
SELECT * FROM orders
WHERE amount > ( SELECT AVG(amount) FROM orders )   ← 안쪽 결과를 기준값으로 사용
                  └──── 서브쿼리: 평균 금액 ────┘
```

쓰이는 위치에 따라:

- **`WHERE`절 서브쿼리** — 조건의 비교 대상으로 (위 예시)
- **`FROM`절 서브쿼리(인라인 뷰)** — 서브쿼리 결과를 *하나의 임시 테이블*처럼 사용

▶️ **코드 실행하기 · 코드 셀 19 [C19]**  
— **전체 주문의 평균 금액보다 큰** 주문만 찾습니다(`WHERE`절 서브쿼리).

In [ ]:
-- [C19]
SELECT
  order_id,
  amount
FROM project_name.dataset_name.orders
WHERE
  amount > (
    SELECT
      AVG(amount)
    FROM project_name.dataset_name.orders
  )
ORDER BY
  amount DESC

▶️ **코드 실행하기 · 코드 셀 20 [C20]**  
— **미국 고객의 번호 목록을 먼저 구해**, 그 목록에 속하는 주문만 가져옵니다(`IN` + 서브쿼리).

In [ ]:
-- [C20]
SELECT
  order_id,
  customer_id,
  amount
FROM project_name.dataset_name.orders
WHERE
  customer_id IN (
    SELECT
      customer_id
    FROM project_name.dataset_name.customers
    WHERE
      country = 'USA'
  )

> 👀 **결과읽기 — [C19]·[C20]**  
> 첫 쿼리는 _괄호 안에서 평균을 먼저 구한 뒤_ 그보다 큰 주문을 찾습니다. 두 번째는 *미국 고객의 번호 목록*을 먼저 구해, 그 목록에 속하는 주문만 가져옵니다. 질문을 단계로 쪼갠 겁니다.

▶️ **코드 실행하기 · 코드 셀 21 [C21]**  
— 고객별 총 구매액을 **`FROM`절 서브쿼리(인라인 뷰)로** 먼저 만든 뒤, **총 구매액이 20만원 이상인 고객만** 남깁니다.

In [ ]:
-- [C21]
SELECT
  *
FROM (
  SELECT
    customer_id,
    SUM(amount) AS total_amount
  FROM project_name.dataset_name.orders
  WHERE
    amount IS NOT NULL
  GROUP BY
    customer_id
) AS customer_total
WHERE
  total_amount >= 200000
ORDER BY
  total_amount DESC

> 👀 **결과읽기 — [C21]**  
> 괄호 안 서브쿼리가 _"고객별 총구매액"_ 이라는 임시 표를 만들고, 바깥 쿼리가 그 표에서 20만 이상만 골랐습니다. 복잡한 집계를 단계로 나누는 강력한 패턴입니다. (다음 시간에 배울 CTE가 이걸 더 읽기 좋게 만들어 줍니다.)

> 📌 **실무 연결하기 — 기준을 구하고 비교하기**  
> 서브쿼리는 _평균 이상 거래 탐지_(금융), _상위 사용자만 필터_(마케팅), _기준치 초과 센서_(제조)처럼 "먼저 기준을 구하고 → 비교"하는 모든 곳에 쓰입니다.

## ⌨️ 백문이 불여일타 (7)

```
[문제] 한국(Korea) 고객의 주문만 골라, 주문번호·금액을 금액 큰 순으로 조회합니다.
(WHERE절 서브쿼리 + IN 사용)
```

In [ ]:
-- [C22] ⌨️ 백문이 불여일타 (7) — 서브쿼리로 한국 고객 주문 조회

-- 여기에 SQL 쿼리를 작성하세요.

SELECT order_id, amount
FROM project_name.dataset_name.orders
WHERE customer_id IN (
  SELECT customer_id
  FROM project_name.dataset_name.customers
  WHERE country = 'Korea'
  )
  AND amount IS NOT NULL
ORDER BY amount DESC

<details>
<summary>(펼치기) 🔑 힌트</summary>

- 먼저 괄호 안 서브쿼리로 한국 고객의 `customer_id` 목록을 구합니다.
- 바깥 쿼리에서 `WHERE customer_id IN (...)`으로 그 목록에 속한 주문만 남깁니다.
- 정렬 전에 `amount IS NOT NULL` 조건도 함께 넣습니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  order_id,
  amount
FROM project_name.dataset_name.orders
WHERE
  customer_id IN (
    SELECT
      customer_id
    FROM project_name.dataset_name.customers
    WHERE
      country = 'Korea'
  )
  AND amount IS NOT NULL
ORDER BY
  amount DESC
```

</details>

## 스칼라 서브쿼리 — 값 하나를 `SELECT`에 끼워 넣기

서브쿼리가 _값 하나_(스칼라)를 돌려주면, `SELECT`절에 끼워 넣어 "각 행 옆에 전체 기준값"을 붙일 수도 있습니다.

▶️ **코드 실행하기 · 코드 셀 23 [C23]**  
— **금액이 입력된** 주문 상위 5건 옆에, **전체 평균 금액과 그 평균과의 차이**를 값 하나로 끼워 넣습니다(스칼라 서브쿼리).

In [ ]:
-- [C23]
SELECT
  order_id,
  amount,
  (
    SELECT
      ROUND(AVG(amount), 0)
    FROM project_name.dataset_name.orders
  ) AS 전체평균,
  amount - (
    SELECT
      AVG(amount)
    FROM project_name.dataset_name.orders
  ) AS 평균과의차이
FROM project_name.dataset_name.orders
WHERE
  amount IS NOT NULL
ORDER BY
  amount DESC
LIMIT 5

> 👀 **결과읽기 — [C23]**  
> 괄호 안 서브쿼리가 *전체 평균*이라는 값 하나를 돌려주고, 그 값이 모든 행에 똑같이 붙습니다. (이후 배울 윈도우 함수로도 같은 일을 더 우아하게 할 수 있습니다.)

> ➕ **심화 학습하기 — 상관 서브쿼리**  
> 서브쿼리가 *바깥 행마다 다른 값*을 참조하면 '상관 서브쿼리'라고 합니다. 예: 각 주문에 대해 *그 고객의 평균 주문액*을 구하는 경우. 강력하지만 느릴 수 있어, 보통은 `JOIN`이나 윈도우 함수로 대체합니다. (이번 모듈 필수 범위는 아닙니다.)

## 🚦 짚고 넘어가기

1. 바깥 쿼리가 서브쿼리의 결과를 조건으로 사용한다면, 논리적으로 어떤 결과가 먼저 필요합니까?
2. `WHERE`절 서브쿼리와 `FROM`절 서브쿼리(인라인 뷰)는 각각 어떻게 쓰입니까?
3. 서브쿼리로 "평균보다 큰 행"을 어떻게 찾습니까?

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

1. 안쪽 서브쿼리의 결과가 먼저 필요합니다. 다만 이는 결과를 이해하기 위한 논리적 순서이며, 실제 실행 순서는 DBMS의 최적화기가 바꿀 수 있습니다.
2. `WHERE`절 서브쿼리는 조건에 쓸 값이나 목록을 만들어 주고, `FROM`절 서브쿼리(인라인 뷰)는 임시 표를 만들어 바깥 쿼리가 그 표를 다시 조회하게 합니다.
3. 서브쿼리로 평균(`SELECT AVG(amount) ...`)을 먼저 구하고, 바깥 쿼리에서 `WHERE amount > (그 서브쿼리)`처럼 비교합니다.

</details>

> ⏭️ **다음 학습 예고**  
> 서브쿼리가 깊어지면 읽기 어려워집니다. 다음 시간엔 이를 *이름 붙인 단계*로 푸는 **CTE(`WITH`)** 와, 행을 유지한 채 순위·누적을 구하는 **윈도우 함수**를 배웁니다.

# Part 6. 세 테이블 이상 결합과 `EXISTS`

실무 질문은 테이블 두 개로 끝나는 일이 드뭅니다. _"어느 카테고리 상품이 어느 나라 고객에게 팔렸나?"_ 를 답하려면 `customers` · `orders` · `order_items` · `products` **네 개**를 이어야 합니다.

> ❓ **질문하기**  
> 테이블이 셋 이상일 때 `JOIN`을 어떻게 이어 붙이고, "존재하는지만" 확인할 때는 무엇을 씁니까?

## 🧩 개념 잡기

`JOIN`은 **왼쪽부터 차례로 이어 붙입니다.** 두 개를 붙여 만든 결과에 세 번째를 또 붙이는 식입니다.

```
customers ──(customer_id)── orders ──(order_id)── order_items ──(product_id)── products
   고객                       주문                    주문 상세                    상품
```

각 `JOIN`마다 **연결 열쇠(`ON`)가 하나씩** 필요합니다. 열쇠를 빠뜨리면 모든 행이 서로 곱해지는 사고(카테시안 곱)가 납니다.

▶️ **코드 실행하기 · 코드 셀 24 [C24]**  
— `customers`·`orders`·`order_items`·`products` **네 테이블을 모두 이어**, **국가가 입력된** 고객의 국가 × 카테고리별 매출을 구합니다.

In [ ]:
-- [C24]
SELECT
  c.country,
  p.category,
  SUM(oi.quantity * oi.unit_price) AS 매출
FROM project_name.dataset_name.customers AS c
JOIN project_name.dataset_name.orders AS o
  ON c.customer_id = o.customer_id
JOIN project_name.dataset_name.order_items AS oi
  ON o.order_id = oi.order_id
JOIN project_name.dataset_name.products AS p
  ON oi.product_id = p.product_id
WHERE
  c.country IS NOT NULL
GROUP BY
  c.country,
  p.category
ORDER BY
  매출 DESC

> 👀 **결과읽기 — [C24]**  
> `JOIN ... ON ...`이 세 번 이어졌습니다. 읽는 순서는 위에서 아래로 — _고객에 주문을 붙이고, 거기에 주문 상세를 붙이고, 다시 상품을 붙인다_. 별칭(`c`·`o`·`oi`·`p`)을 짧게 두면 긴 조인도 읽기 쉬워집니다.

> ⚠️ **주의하기 — `ON`을 빠뜨리면 큰일 납니다**  
> 연결 조건 없이 두 테이블을 조인하면 15명 × 30주문 = **450행**이 만들어집니다. 결과 행 수가 갑자기 폭증하면 가장 먼저 `ON`을 의심합니다.

## `EXISTS` — 값이 아니라 '존재 여부'만 물을 때

_"주문한 적이 있는 고객"_ 을 찾는다고 가정합니다. `JOIN`을 쓰면 주문이 5건인 고객은 **5번 중복**됩니다. 중복을 없애려면 `DISTINCT`를 또 붙여야 합니다.

이럴 때 `EXISTS`가 깔끔합니다. **"짝이 하나라도 있으면 참"** 만 판정하고 행을 늘리지 않습니다.

▶️ **코드 실행하기 · 코드 셀 25 [C25]**  
— **주문한 적이 있는 고객만** 찾습니다 — `EXISTS`는 짝이 하나라도 있으면 참이라 행이 늘어나지 않습니다.

In [ ]:
-- [C25]
SELECT
  customer_id,
  name,
  country
FROM project_name.dataset_name.customers AS c
WHERE
  EXISTS (
    SELECT 1
    FROM project_name.dataset_name.orders AS o
    WHERE o.customer_id = c.customer_id
  )
ORDER BY
  customer_id

> 👀 **결과읽기 — [C25]**  
> 안쪽 쿼리의 `SELECT 1`은 _"무엇을 가져올지는 상관없고, 행이 있기만 하면 된다"_ 는 관용적 표현입니다. 바깥 `c.customer_id`를 안쪽에서 참조하는 것이 핵심입니다 — 고객 한 명씩 짚어가며 "이 사람 주문이 있나?"를 묻습니다.

> 💡 **핵심짚기 — `NOT EXISTS`로 뒤집으면**  
> _"주문한 적 없는 고객"_ 이 됩니다. `LEFT JOIN` + `IS NULL`로도 같은 답을 얻을 수 있는데, 의도가 더 잘 드러나는 쪽을 고르면 됩니다.

> 📌 **실무 연결하기 — 존재 여부만 묻기**  
> _"진료 이력이 있는 환자"_(헬스케어), _"거래가 발생한 계좌"_(금융), _"응답이 있는 설문 대상"_(리서치) — 존재 여부만 묻는 질문은 어디에나 있습니다.

## ➕ 심화 학습하기 — 나머지 조인 종류

지금까지 `INNER`·`LEFT`·`RIGHT`를 배웠습니다. 실무에서 가끔 만나는 나머지 셋도 이름만 알아 두세요.

| 종류              | 하는 일                        | 쓰는 순간                                                  |
| ----------------- | ------------------------------ | ---------------------------------------------------------- |
| `FULL OUTER JOIN` | 양쪽 모두, 짝 없는 행까지 전부 | 두 시스템의 데이터를 대조해 **어느 쪽에만 있는지** 찾을 때 |
| `CROSS JOIN`      | 모든 조합을 곱함 (`ON` 없음)   | 날짜 × 상품처럼 **빈칸 없는 격자**를 만들 때               |
| `SELF JOIN`       | 같은 테이블을 자기 자신과 조인 | 직원-상사, 카테고리-상위카테고리 같은 **계층 구조**        |

```sql
-- SELF JOIN 예시 — 같은 국가의 다른 고객 짝 짓기
SELECT
  a.name AS 고객A,
  b.name AS 고객B,
  a.country
FROM customers AS a
JOIN customers AS b
  ON a.country = b.country AND a.customer_id < b.customer_id
```

> 💡 `a.customer_id < b.customer_id` 조건이 없으면 (김민준-이서연)과 (이서연-김민준)이 **둘 다** 나오고, 자기 자신과의 짝까지 생깁니다.

## 🚦 짚고 넘어가기

1. 테이블 셋을 조인할 때 `ON`은 몇 개 필요합니까?
2. `JOIN`으로 "주문한 적 있는 고객"을 구하면 무엇이 문제입니까?
3. `EXISTS` 안의 `SELECT 1`은 무슨 뜻입니까?

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

1. **두 개**입니다. 조인을 한 번 할 때마다 연결 조건이 하나씩 필요하므로, 테이블 N개를 이으려면 `ON`이 N−1개 필요합니다.
2. 주문이 여러 건인 고객이 **그 건수만큼 중복**돼 나옵니다. `DISTINCT`로 지울 수 있지만, 애초에 행을 늘리지 않는 `EXISTS` 쪽이 의도가 분명합니다.
3. _"가져올 값은 중요하지 않고 행이 존재하는지만 본다"_ 는 뜻입니다. `SELECT *`나 `SELECT customer_id`로 써도 결과는 같습니다.

</details>

> ⏭️ **다음 학습 예고**  
> 오늘 배운 조인과 서브쿼리를 5문항으로 반복합니다.

# 📒 연습 문제 — 조인과 서브쿼리 (5문항)

테이블을 잇는 연습입니다. 각 문제마다 **어떤 테이블을 무슨 열쇠로 이을지** 먼저 종이에 그려 보고 쿼리를 쓰면 훨씬 쉽습니다.

**풀이 방법:** 문제를 먼저 풀고 실행한 뒤 바로 아래의 예시 답안을 펼쳐 비교합니다. 막히면 앞 Part의 예제 셀을 다시 확인합니다.

In [ ]:
-- [C26] 📒 연습 문제 — 조인과 서브쿼리 (1/5)

-- 문제: 고객의 국가(country)별 주문 건수를 구하고, 많은 순으로 정렬합니다.

-- 아래에 SQL 쿼리를 작성하세요.

SELECT c.country, COUNT(*) AS 주문건수
FROM project_name.dataset_name.orders AS o
JOIN project_name.dataset_name.customers AS c
  ON o.customer_id = c.customer_id
WHERE c.country IS NOT NULL
GROUP BY c.country
ORDER BY 주문건수 DESC;

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
-- [1] 조인 후 그룹 집계
SELECT
  c.country,
  COUNT(*) AS 주문건수
FROM project_name.dataset_name.orders AS o
JOIN project_name.dataset_name.customers AS c
  ON o.customer_id = c.customer_id
WHERE c.country IS NOT NULL
GROUP BY c.country
ORDER BY 주문건수 DESC;
```

</details>

In [ ]:
-- [C27] 📒 연습 문제 — 조인과 서브쿼리 (2/5)

-- 문제: orders·order_items·products 세 테이블을 조인해 주문번호·상품명·수량·단가를 조회합니다.

-- 아래에 SQL 쿼리를 작성하세요.

SELECT o.order_id, p.product_id, oi.quantity, oi.unit_price
FROM project_name.dataset_name.orders AS o
JOIN project_name.dataset_name.order_items AS oi
  ON o.order_id = oi.order_id
JOIN project_name.dataset_name.products AS p
  ON oi.product_id = p.product_id
ORDER BY o.order_id

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
-- [2] 세 테이블 조인 — ON 이 두 개
SELECT
  o.order_id,
  p.product_name,
  oi.quantity,
  oi.unit_price
FROM project_name.dataset_name.orders AS o
JOIN project_name.dataset_name.order_items AS oi
  ON o.order_id = oi.order_id
JOIN project_name.dataset_name.products AS p
  ON oi.product_id = p.product_id
ORDER BY o.order_id;
```

</details>

In [ ]:
-- [C28] 📒 연습 문제 — 조인과 서브쿼리 (3/5)

-- 문제: 주문이 한 건도 없는 고객의 이름과 국가를 찾으세요. (LEFT JOIN 활용)

-- 아래에 SQL 쿼리를 작성하세요.

SELECT
  c.name,
  c.country
FROM project_name.dataset_name.customers AS c
LEFT JOIN project_name.dataset_name.orders AS o
  ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL;

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
-- [3] 짝이 없는 행 찾기 — LEFT JOIN 후 NULL 검사
SELECT
  c.name,
  c.country
FROM project_name.dataset_name.customers AS c
LEFT JOIN project_name.dataset_name.orders AS o
  ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL;
```

**확인할 점**

- **`WHERE`의 위치** — `LEFT JOIN`으로 붙인 뒤 `WHERE o.order_id IS NULL`로 걸러야 합니다. 이 조건을 `ON`에 넣으면 필터가 조인 _전에_ 걸려 결과가 달라집니다.

</details>

In [ ]:
-- [C29] 📒 연습 문제 — 조인과 서브쿼리 (4/5)

-- 문제: 카테고리별 총매출(수량 × 단가)을 구하고 매출이 큰 순으로 정렬합니다.

-- 아래에 SQL 쿼리를 작성하세요.

SELECT p.category, SUM( oi.quantity * oi.unit_price) AS 매출액
FROM project_name.dataset_name.order_items AS oi
JOIN project_name.dataset_name.products AS p
  ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY 매출액 DESC ;

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
-- [4] 카테고리별 매출
SELECT
  p.category,
  SUM(oi.quantity * oi.unit_price) AS 매출
FROM project_name.dataset_name.order_items AS oi
JOIN project_name.dataset_name.products AS p
  ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY 매출 DESC;
```

</details>

In [ ]:
-- [C30] 📒 연습 문제 — 조인과 서브쿼리 (5/5)

-- 문제: 전체 주문의 평균 금액보다 큰 주문만 조회합니다. (서브쿼리 활용)

-- 아래에 SQL 쿼리를 작성하세요.

SELECT order_id, amount
FROM project_name.dataset_name.orders 
WHERE amount > (
  SELECT AVG(amount)
  FROM project_name.dataset_name.orders
)
ORDER BY amount DESC;

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
-- [5] 스칼라 서브쿼리를 조건으로
SELECT
  order_id,
  amount
FROM project_name.dataset_name.orders
WHERE amount > (
  SELECT AVG(amount) FROM project_name.dataset_name.orders
)
ORDER BY amount DESC;
```

**확인할 점**

- **서브쿼리의 `NULL`** — `AVG(amount)`는 `NULL`을 자동으로 제외하고 평균을 냅니다. 취소 주문 3건은 평균 계산에 들어가지 않습니다.

</details>

# 🧪 종합 실습 — 월별·카테고리별 매출 TOP 10

오늘 배운 `JOIN`과 서브쿼리를 모아, 모두몰의 **"매출 TOP 리포트"** 를 만듭니다. 여러 테이블을 이어야만 답할 수 있는 질문들입니다.

**질문 1.** 카테고리별 매출(수량 × 단가의 합)을 구해, 매출이 큰 카테고리 순으로 보여주세요. (`order_items` + `products`)

In [ ]:
-- [C31] 🧪 종합 실습 — 월별·카테고리별 매출 TOP 10 (1/4)

-- 아래에 SQL 쿼리를 작성하세요.

SELECT
  p.category AS 카테고리,
  SUM(oi.unit_price * oi.quantity) AS 매출
FROM project_name.dataset_name.order_items AS oi
JOIN project_name.dataset_name.products AS p
  ON oi.product_id =p.product_id
GROUP BY 카테고리
ORDER BY 매출 DESC ;

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  p.category AS 카테고리,
  SUM(oi.quantity * oi.unit_price) AS 매출
FROM project_name.dataset_name.order_items AS oi
JOIN project_name.dataset_name.products AS p
  ON oi.product_id = p.product_id
GROUP BY
  카테고리
ORDER BY
  매출 DESC
```

</details>

**질문 2.** 고객 이름별 총 구매액 TOP 5를 구합니다. (`orders` + `customers`)

In [ ]:
-- [C32] 🧪 종합 실습 — 월별·카테고리별 매출 TOP 10 (2/4)

-- 아래에 SQL 쿼리를 작성하세요.

SELECT
  c.name AS 고객이름,
  c.grade AS 등급,
  SUM(o.amount) AS 총구매액
FROM project_name.dataset_name.orders AS o 
JOIN project_name.dataset_name.customers AS c
  ON o.customer_id = c.customer_id
WHERE o.amount IS NOT NULL
GROUP BY 고객이름, 등급
ORDER BY 총구매액 DESC
LIMIT 5;

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  c.name AS 고객명,
  c.grade AS 등급,
  SUM(o.amount) AS 총구매액
FROM project_name.dataset_name.orders AS o
JOIN project_name.dataset_name.customers AS c
  ON o.customer_id = c.customer_id
WHERE
  o.amount IS NOT NULL
GROUP BY
  고객명,
  등급
ORDER BY
  총구매액 DESC
LIMIT 5
```

</details>

**질문 3.** 월별 × 카테고리별 매출을 구해, 매출 상위 10개 (월, 카테고리) 조합을 보여주세요. 그리고 카테고리별 매출을 그래프로 그립니다.

In [ ]:
-- [C33] 🧪 종합 실습 — 월별·카테고리별 매출 TOP 10 (3/4)

-- 아래에 SQL 쿼리를 작성하세요.

SELECT
  date_trunc('month', o.order_date) AS 월,
  p.category AS 카테고리,
  SUM(oi.quantity * oi.unit_price) AS 매출
FROM project_name.dataset_name.orders AS o
JOIN project_name.dataset_name.order_items AS oi
  ON o.order_id = oi.order_id
JOIN project_name.dataset_name.products AS p
  ON oi.product_id = p.product_id
GROUP BY 월, 카테고리
ORDER BY 매출 DESC
LIMIT 10

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  date_trunc('month', o.order_date) /* BigQuery: DATE_TRUNC(o.order_date, MONTH) */ AS 월,
  p.category AS 카테고리,
  SUM(oi.quantity * oi.unit_price) AS 매출
FROM project_name.dataset_name.orders AS o
JOIN project_name.dataset_name.order_items AS oi
  ON o.order_id = oi.order_id
JOIN project_name.dataset_name.products AS p
  ON oi.product_id = p.product_id
GROUP BY
  월,
  카테고리
ORDER BY
  매출 DESC
LIMIT 10
```

</details>

In [ ]:
-- [C34] 🧪 종합 실습 — 월별·카테고리별 매출 TOP 10 (4/4)

-- 아래에 SQL 쿼리를 작성하세요.
SELECT
  p.category AS category,
  SUM(oi.quantity * oi.unit_price) AS revenue
FROM project_name.dataset_name.order_items AS oi
JOIN project_name.dataset_name.products AS p
  ON oi.product_id = p.product_id
GROUP BY
  category
ORDER BY
  revenue DESC

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  p.category AS category,
  SUM(oi.quantity * oi.unit_price) AS revenue
FROM project_name.dataset_name.order_items AS oi
JOIN project_name.dataset_name.products AS p
  ON oi.product_id = p.product_id
GROUP BY
  category
ORDER BY
  revenue DESC
```

</details>

> 📊 이 결과는 LMS에서는 표로 확인하고, BigQuery에서는 콘솔 차트(또는 Looker Studio)로 시각화할 수 있습니다.

> 👀 **결과읽기 — [C33]·[C34]**  
> 세 테이블(`orders`·`order_items`·`products`)을 이어야만 "월별·카테고리별 매출"을 구할 수 있습니다. 흩어진 데이터를 합쳐 비로소 비즈니스 질문에 답한 겁니다.

# 🎯 오늘의 퀴즈

배운 내용을 잠깐 확인해보겠습니다. 틀려도 괜찮습니다!

## 개념 퀴즈

1. 각 행을 유일하게 구분하는 열을 (PK / FK), 다른 표의 그것을 가리키는 열을 (PK / FK)라고 한다.
2. `INNER JOIN`과 `LEFT JOIN` 중, "짝이 없어도 왼쪽 행을 남기는" 것은?
3. `JOIN`은 테이블을 (옆 / 세로)으로, `UNION`은 (옆 / 세로)으로 붙인다.
4. "평균보다 큰 주문"을 구할 때 서브쿼리는 무엇을 먼저 계산합니까?

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

1. PK / FK
2. **LEFT `JOIN`**
3. `JOIN` = 옆, `UNION` = 세로
4. 괄호 안 서브쿼리가 **평균값**을 먼저 계산하고, 바깥 쿼리가 그 값과 비교합니다.
</details>

## 코드 퀴즈

**문제:** `orders`·`customers`를 합쳐 **등급(`grade`)별 총 매출**을 구합니다(금액 있는 것만). 매출 큰 순으로.

아래 셀에 직접 작성한 뒤, 예시 답안과 비교합니다.

In [ ]:
-- [C35] 코드 퀴즈 — 등급별 총 매출

-- 여기에 SQL 쿼리를 작성하세요.

SELECT
  c.grade AS 등급,
  SUM(o.amount) AS 총매출
FROM project_name.dataset_name.orders AS o
JOIN project_name.dataset_name.customers AS c
  ON o.customer_id = c.customer_id
WHERE o.amount IS NOT NULL
GROUP BY c.grade
ORDER BY 총매출 DESC;

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  c.grade,
  SUM(o.amount) AS 총매출
FROM project_name.dataset_name.orders AS o
JOIN project_name.dataset_name.customers AS c
  ON o.customer_id = c.customer_id
WHERE
  o.amount IS NOT NULL
GROUP BY
  c.grade
ORDER BY
  총매출 DESC
```

</details>

# 🎓 오늘의 정리

## 오늘 배운 것이 어떻게 이어지는가

```
[Part 1] PK·FK         (테이블을 잇는 열쇠)
   ↓
[Part 2] INNER JOIN    (양쪽에 짝 있는 행 합치기 / 옆으로)
   ↓
[Part 3] LEFT JOIN     (짝 없는 행도 살리기 — 미판매 상품 발견)
   ↓
[Part 4] UNION         (두 목록을 세로로 이어 붙이기)
   ↓
[Part 5] 서브쿼리       (쿼리 안의 쿼리 — 질문을 단계로)
   ↓
[다음 시간] CTE·윈도우 함수 (단계를 이름으로 / 순위·누적)
```

## 한 장 정리 — 오늘의 쿼리 패턴 치트시트

| 하고 싶은 것          | SQL                                                   |
| --------------------- | ----------------------------------------------------- |
| 두 테이블 합치기      | `FROM a JOIN b ON a.key = b.key`                      |
| 짝 없는 행도 남기기   | `FROM a LEFT JOIN b ON ...`                           |
| 짝 없는 행만 찾기     | `LEFT JOIN ... WHERE b.key IS NULL`                   |
| 세로로 합치기         | `SELECT ... UNION [ALL] SELECT ...`                   |
| 조건에 다른 쿼리 결과 | `WHERE x > (SELECT ... )` / `WHERE x IN (SELECT ...)` |
| 집계 결과를 다시 필터 | `FROM (SELECT ... GROUP BY ...) AS t WHERE ...`       |

> 💡 **핵심짚기 — `JOIN`은 옆으로, `UNION`은 세로로**  
> `JOIN`은 옆으로(열), `UNION`은 세로로(행). 둘 다 흩어진 데이터를 하나로 모읍니다.

## 📝 오늘의 과제

**모두몰 매출 TOP 리포트** — 여러 테이블을 이어야 답할 수 있는 분석 질문 5개를 만들고 SQL로 답합니다.

**제출물:**

1. `JOIN`이 필요한 질문 5개 (예: "고객 등급별 매출은?", "카테고리별 베스트셀러는?")
2. 각 질문에 답하는 SQL (`JOIN` 활용)
3. 결과에서 발견한 인사이트 1줄씩

**조건:**

- `INNER JOIN`과 `LEFT JOIN`을 각각 최소 1번 사용
- 서브쿼리(`WHERE`절 또는 `FROM`절)를 최소 1번 사용
- `JOIN` + `GROUP BY`로 그룹별 집계 1개 이상

**평가 기준**

| 항목     | 기준                                   |
| -------- | -------------------------------------- |
| 정확성   | `JOIN` 조건·집계가 의도대로 동작하는가 |
| 적절성   | 질문에 맞는 `JOIN` 종류를 골랐는가     |
| 인사이트 | 합친 데이터에서 시사점을 읽어냈는가    |

**제출:** 개인 GitHub 저장소 main에 커밋·푸시하고 링크를 제출합니다.

## ⏭️ 다음 학습 예고 — SQL 심화: CTE와 윈도우 함수

오늘 서브쿼리로 질문을 단계로 나눠봤습니다. 그런데 서브쿼리가 깊어지면 괄호가 많아져 읽기 힘듭니다.

다음 시간에는 이를 *이름 붙인 단계*로 깔끔하게 푸는 **CTE(`WITH`)** 와, `GROUP BY`처럼 행을 접지 않고 _행을 유지한 채_ 순위·누적·전월대비를 구하는 **윈도우 함수**를 배웁니다. SQL이 한 단계 강력해집니다!

오늘 여러분은 흩어진 데이터를 하나로 잇는, 분석가의 핵심 기술을 익혔습니다.

오늘도 한 걸음, 수고하셨습니다! 🎉

---

<sub>© 2026 모두의연구소(MODULABS). All rights reserved.<br>
기획·제작: 교육퍼실리테이터팀 이진영 (jy.lee@modulabs.co.kr)<br>
본 자료는 생성형 AI를 활용해 제작되었고, 제작자의 검수를 거쳐 완성되었습니다.<br>
무단 복제 및 배포를 금합니다.</sub>